In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path('/content/drive/MyDrive/Logika Ui')
if not ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
HASIL = ROOT / 'KTI_KARHUTLA' / 'hasil'

tabel = pd.read_csv(HASIL / 'tabel_gabungan_nasional_semua_bulan.csv',
                    parse_dates=['tanggal'], usecols=['tanggal', 'GID_2', 'him_n'])
z = np.load(HASIL / 'prob_final_nasional.npz')
print('isi npz:', list(z.keys()))
assert len(z['PD']) == len(tabel), 'urutan baris tidak sama dengan tabel'

tabel['pd'] = z['PD'][:, 1]
tabel['him'] = tabel['him_n'].notna().astype(int)
t = tabel[tabel['tanggal'] > tabel['tanggal'].max() - pd.Timedelta(days=370)]

tgl = [x.strftime('%Y-%m-%d') for x in sorted(t['tanggal'].unique())]
kab = sorted(t['GID_2'].unique())
pv = t.pivot(index='tanggal', columns='GID_2', values='pd').reindex(columns=kab).to_numpy()
hv = t.pivot(index='tanggal', columns='GID_2', values='him').reindex(columns=kab).to_numpy()

isi = {'tgl': tgl, 'kab': kab,
       'pk1d': np.where(np.isnan(pv), -1, np.round(pv * 1000)).astype(int).ravel().tolist(),
       'him': np.where(np.isnan(hv), 0, hv).astype(int).ravel().tolist()}
keluar = HASIL / 'fuwa_data_d.js'
keluar.write_text('window.FUWA_D=' + json.dumps(isi, separators=(',', ':')) + ';', encoding='utf-8')
print('tersimpan:', keluar, '|', len(tgl), 'hari x', len(kab), 'wilayah |', round(keluar.stat().st_size / 1e6, 2), 'MB')
print('rata-rata peluang K1 model D:', round(float(np.nanmean(pv)), 3), '| wilayah-hari tanpa Himawari:', round(1 - float(np.nanmean(hv)), 3))

Mounted at /content/drive
isi npz: ['PC', 'PD', 'PD_kalibrasi']
tersimpan: /content/drive/MyDrive/Logika Ui/KTI_KARHUTLA/hasil/fuwa_data_d.js | 370 hari x 479 wilayah | 1.02 MB
rata-rata peluang K1 model D: 0.238 | wilayah-hari tanpa Himawari: 0.003
